# 환경 관찰과 reset

**이 단원의 목적: VLA 정책을 실행하기 전에, 가상 환경이 정상적으로 열리고 정책이 볼 초기 관측을 만들 수 있는지 확인합니다.**

예상 시간 45분. [전체 안내](README.md). 설명을 읽고 기준 장면 생성 → 비교 실험 → 관찰 TODO 순서로 진행합니다.

## 왜 바로 로봇을 움직이지 않고 장면부터 보나요?

앞의 [03 준비 확인](03_container.ipynb)은 파일·경로·GPU 등 기본 준비 상태를 검사했습니다. 하지만 검사 통과만으로 **가상 카메라 영상이 실제로 생성되는지, 대상 물체가 보이는지** 는 알 수 없습니다. 04에서는 LIBERO 환경을 열고 reset하여 로봇·물체·두 카메라의 초기 관측을 직접 확인합니다.

예를 들어 이후 정책이 그릇을 잡지 못했을 때, 처음부터 영상이 검거나 다른 물체를 보고 있었다면 환경·관측을 먼저 확인해야 합니다. **환경 관찰을 정책 추론과 분리** 하면 어느 단계에서 문제가 생겼는지 좁힐 수 있습니다. 이 단계는 정책을 로드하거나 물체 집기를 시도하지 않습니다.

## 이 실습에서 답할 세 가지 질문

| 확인할 것 | 수행할 실험 | 이후 실습에 필요한 이유 |
|---|---|---|
| 무엇을 보고 시작하는가? | `inspect-01`로 초기 장면을 만들고 두 카메라에서 대상·로봇·가림을 관찰 | 정책에 들어갈 영상의 의미와 시점을 이해 |
| 같은 조건을 다시 요청하면 어떻게 보이는가? | `inspect-02`, 같은 seed와 초기 배치로 같은 카메라끼리 비교 | 조건을 통제한 비교의 기준 마련 |
| 초기 배치만 바꾸면 무엇이 달라지는가? | `inspect-03`, seed 유지·초기 배치 번호만 변경 | 난수 설정과 배치 선택을 구분하고 장면 변화 관찰 |

실행 이름은 저장 폴더를 구분하는 표식입니다. 이름을 바꾸는 것 자체가 초기 배치를 바꾸지는 않습니다. 기존 완료 결과는 재사용하므로 **실제 reset을 새로 실행하려면 사용하지 않은 실행 이름** 이 필요합니다.

## 이후 실습과 어떻게 이어지나요?

**03 준비 검사 → 04 실제 초기 관측 → 05 입력·행동 형식 → 06 행동 예측 → 07 행동 실행·재관측** 순서로 확인 범위를 넓힙니다.

04에서는 두 영상에 무엇이 보이는지 설명합니다. 05에서는 그 영상과 상태가 어떤 배열·순서로 정책에 들어가고 행동 값은 무엇을 뜻하는지 확인합니다. 06은 모델이 행동을 계산하는 단계, 07은 그 행동을 환경에 적용하는 단계입니다. 후속 실행은 자신의 조건으로 환경을 초기화하므로 04의 PNG를 무조건 정책 입력으로 읽는 흐름은 아닙니다.

**완료 기준:** 실제 관찰한 대상·시점 차이·같은 조건의 재현 여부·다른 배치의 차이를 파일명과 함께 `notes`에 남깁니다. 정책의 작업 성공은 이 단원의 완료 조건이 아닙니다.


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [5일차 실행 안내](README.md)를 참고합니다.

**5일차 환경:** `nepes-course:physical-ai-vla-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 같은 날에는 준비된 개인 VLA 환경을 유지합니다.

**실행 전 확인 — 컨테이너 내부 Terminal:**

```bash
cd /workspace
/opt/course/bin/python 10.VLA/scripts/preflight.py
```

Python 커널도 `/opt/course/bin/python`을 사용합니다. 강사 배포 `Data/models/smolvla-libero-probe`(LIBERO용 정책 revision `31d453f7edd78c839a8bbc39744a292686daf0de`), `Data/cache/smolvla-probe/huggingface`, `Data/cache/smolvla-probe/libero`와 LIBERO 자산이 필요합니다. 파일 누락은 호스트 개인 Data의 마운트·배포본을 확인합니다. 수강생이 `pip install`, Docker build, 모델 다운로드를 하지 않습니다. CUDA/EGL 문제는 전용 이미지·NVIDIA runtime과 다른 GPU 작업을 확인합니다.

**실행과 저장:** 04부터 08까지의 노트북의 `SESSION = 'session-01'`은 같은 회차이며 결과는 `Data/logs/day5/session-01/<실행이름>/`에 남습니다. 새 비교 전체는 모든 후속 노트북의 SESSION을 함께 `session-02` 등으로 바꿉니다. `run_lesson`은 동일 조건의 `completed` 결과를 읽고 없을 때 실행합니다. 미완료·조건 불일치는 기존 파일을 삭제하거나 성공값으로 바꾸지 말고 새 회차/실행 이름을 사용합니다. JSON은 있는데 필요한 PNG/NPZ/NPY/영상이 없으면 부분 손상으로 보고 새 회차에서 다시 생성합니다.

**한 번에 한 정책 프로세스만 실행합니다.** 이전 실행이 끝난 뒤 다음 셀을 실행하고 다른 노트북의 infer/rollout을 동시에 시작하지 않습니다. 실행 중단 후에는 해당 Terminal이나 실행 로그에서 자식 프로세스까지 종료됐는지 확인합니다. `status=completed`는 절차 완료, rollout의 `success=true`는 환경 과제 성공이며 inspect/infer의 `success=false`는 아직 작업을 하지 않았다는 뜻입니다.

## MuJoCo 화면 보기: 로컬·원격 공통 noVNC

<!-- procedure:novnc-1744bd:start -->
![noVNC의 정상 대기와 완료](../교재그림/절차/novnc.svg)

03의 준비 검사와 05의 저장 배열 읽기는 새 실험 창을 열지 않습니다. 기존 완료 결과 재사용에도 새 창이 없습니다.
<!-- procedure:novnc-1744bd:end -->


JupyterLab과 **개인 시뮬레이션 화면을 브라우저 탭 두 개로 열어둡니다.** [README의 최초 접속 안내](README.md#novnc-접속)를 따르세요. 기본 배정은 사용자 A: Jupyter 8890 / 화면 8990, 사용자 B: Jupyter 8891 / 화면 8991입니다. 본인 배정 포트를 사용하며 원격은 두 포트의 SSH 터널을 유지합니다. noVNC에서 Connect를 누르고 암호 `nvidia`를 입력하세요. 처음에는 빈 바탕 화면이 정상이며 실험을 실행하면 MuJoCo 창이 나타납니다.

**이 노트북에서는 초기 장면만 확인합니다.** 정책 로드나 행동 적용을 하지 않으므로 로봇이 움직이지 않는 것이 정상입니다. 각 inspect 실행의 완료 창을 닫은 다음 저장 이미지로 비교합니다.

설정 셀의 `GUI`는 자동으로 켜집니다. 창 없이 시간 비교를 할 때는 `GUI = False`로 바꾸고 설정 셀을 다시 실행합니다. CLI 대체 실행은 `run_episode.py`에 `--gui`를 붙입니다. 노트북과 CLI를 동시에 실행하지 않습니다.

환경 초기화가 끝난 뒤 실제 초기 장면에서 창이 열립니다. 초기화 중 오류는 노트북 로그에서 확인합니다. 화면에는 실행 이름·모드·현재 상태·스텝·추론 횟수·적용 길이·직전 추론 초가 표시됩니다.

| 화면 상태 | 의미 |
|---|---|
| `INITIAL OBSERVATION` | 초기 관측 준비 완료 |
| `MODEL LOADING` | 모델 로드 중, 환경은 대기 |
| `INFERENCE` | 행동 계산 중, 환경은 대기 |
| `ACTION EXECUTION` | 예측 명령을 환경에 적용 중 |
| `FINISHED` | 종료 이유와 성공 여부 표시. 마지막 장면이며 더 움직이지 않음 |
| `INCOMPLETE / ERROR` | 미완료. 노트북의 원인 로그 확인 |

**정상 종료 화면을 관찰한 뒤 창을 닫으면 셀이 끝납니다.** 결과와 영상은 종료 화면 대기 전에 저장됩니다. 실행 도중 창을 닫으면 다음 안전한 처리 지점에서 사용자 중단으로 기록합니다. GPU 추론·모델 로딩 중에는 그 작업이 반환될 때까지 중단 반영이 늦을 수 있습니다. 창 조작은 표시용 복사본에만 적용되어 정책 카메라와 실제 실험 상태를 바꾸지 않습니다.

**기존 완료 결과를 읽으면 새 GUI는 열리지 않습니다.** 새 시연은 사용하지 않은 실행 이름으로 수행하고 변수·파일 경로도 함께 맞춥니다. GUI 사용 여부는 `result.json`의 `gui_enabled`에 남습니다. 화면 표시 비용이 있으므로 GUI 실행과 비GUI 실행의 시간을 같은 조건처럼 비교하지 않습니다.


**브라우저 탭을 닫아도 실험은 종료되지 않습니다.** 다시 같은 주소로 접속하면 진행 중 화면으로 돌아옵니다. FINISHED 후 화면 안의 MuJoCo 창을 닫아야 셀이 끝납니다. noVNC 연결과 탭은 다음 실험까지 유지하세요. 다른 학생 화면이 보이면 포트를 확인합니다.


## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **예제·TODO·검증** → [5일차 슬라이드 28쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s024) · [배포 PDF 28쪽](../슬라이드/5일차.pdf#page=28)에서 31쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 선행 정보와 이 노트북의 입력·산출물

[03_container.ipynb의 「VLA 실습 준비 확인」](03_container.ipynb)에 전용 이미지·Data·커널 준비가 있으며 시작 절에 같은 명령을 반복 제공했습니다. [02_simulation_stack.ipynb의 도구 역할 설명](02_simulation_stack.ipynb)에서 LIBERO가 초기 배치와 성공조건을 정하고 MuJoCo가 두 가상 영상을 만드는 구조를 배웠습니다.

필요한 내용은 `reset`이 초기 관측을 만들고 `step(action)`이 행동 하나를 적용한다는 점입니다. 기본은 libero_spatial task0, seed7, initial_state=0입니다. 아래 설정 셀은 ROOT/SESSION/RUNS/run_lesson을 다시 정의하고 `inspect-01`이 없으면 정책 로드 없이 실제 reset을 수행합니다. 이전 노트북 산출물을 읽지 않습니다.

`RUNS/inspect-01/`에는 `result.json`(task·초기번호·seed·완료상태), `initial-observation.npz`(raw RGB 영상2와 상태8), `image.png`,`image2.png`(가로·세로 반전한 표시 영상)가 저장됩니다. 비교할 때 NPZ 원본과 PNG의 방향 차이를 시점 차이와 혼동하지 마세요.

## 개념

reset은 새 에피소드의 초기 관측을 얻는 동작이고 step은 행동 하나를 적용하는 동작입니다. 기준은 libero_spatial의 task 0입니다. 아래 실행은 정책을 로드하지 않고 실제 환경을 reset하여 카메라와 상태를 저장합니다.

seed는 난수 재현 설정이며 초기 배치 번호가 아닙니다. 초기 배치는 initial_state로 선택합니다. LeRobot 생성자의 episode_index를 사용하고 각 프로세스에서 한 번 reset합니다. 같은 번호·seed를 사용하면 같은 초기조건을 요청합니다. 동일 seed도 소프트웨어·장치 변화까지 완전한 재현을 보장하지 않습니다.

### 단계별 활동

진행: reset/step 설명10분 → 두 시점 관찰15분 → 같은 초기조건 재실행10분 → 다른 초기조건 비교10분. inspect-03 이름으로 initial_state=1을 실행해 그릇 위치를 비교하세요. 같은 task 문장이 유지되는지도 확인합니다. 카메라가 달라도 같은 물리 세계를 보는지 물체를 대응시켜 보세요.

In [ ]:
from pathlib import Path
import sys, subprocess, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '10.VLA').is_dir())
SCRIPTS = ROOT / '10.VLA/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, lesson_runs, run_or_read_lesson
# 04~08 모두 같은 회차를 사용합니다. 전체 비교를 새로 할 때 session-02 등으로 함께 변경하세요.
GUI = os.environ.get('COURSE_VLA_GUI', '0') == '1'  # 개인 noVNC 화면 사용
SESSION = 'session-01'
RUNS = lesson_runs(DATA, SESSION)
RUNS.mkdir(parents=True, exist_ok=True)
def run_lesson(mode, name, seed=7, execute=10, steps=280, initial_state=0):
    return run_or_read_lesson(SCRIPTS, RUNS, mode, name, seed, execute, steps, initial_state, gui=GUI)


## 예제 — 실행 중 표시와 완료 창 구분하기

<!-- procedure:lesson-40-9772fb:start -->
![초기 장면 생성 후 결과 읽기](../교재그림/절차/lesson-40.svg)

새 이름만 적으면 파일이 생기지 않습니다. inspect-02도 생성 셀 전체를 실행한 다음 읽으세요.
<!-- procedure:lesson-40-9772fb:end -->


아래 셀은 **환경 초기화 → 관측 저장 → noVNC에서 마지막 장면 관찰 → 창 닫기 → 노트북 이미지 표시** 순서로 진행합니다.

1. 처음에는 환경을 초기화하므로 시간이 걸립니다. 이 장비의 한 실행에서는 약 46초였으며, 고정된 대기 시간은 아닙니다.
2. noVNC의 MuJoCo 화면에 `FINISHED`가 보이면 **초기화와 결과 저장은 이미 완료** 된 상태입니다. 장면을 관찰하도록 창을 유지하므로 노트북의 `[*]` 표시와 경과 시간은 계속됩니다. 이때는 초기화가 계속되는 것이 아닙니다.
3. 관찰을 마쳤으면 **화면 안의 MuJoCo 창 오른쪽 위 × 버튼** 을 누릅니다. 브라우저 탭을 닫는 것은 연결만 끊으며 셀을 끝내지 않습니다.
4. 창을 닫으면 셀이 완료되고 아래 두 카메라 이미지와 결과가 표시됩니다. TODO의 비교 실행도 같은 순서입니다.

`FINISHED`가 없거나 오류가 보이면 노트북 로그를 확인합니다. 실행 중 창을 닫는 것은 중단이며, 완료 후 창을 닫는 것과 구분합니다. 기존 완료 기록을 읽은 경우에는 새 창이 열리지 않고 셀이 바로 끝납니다.


In [ ]:
result = run_lesson('inspect', 'inspect-01')
from IPython.display import display, Image
display(Image(filename=str(RUNS/'inspect-01/image.png')))
display(Image(filename=str(RUNS/'inspect-01/image2.png')))
result

## TODO — 같은 초기조건으로 다시 관찰하기

새 실행 이름으로 같은 초기조건을 요청하고, 두 카메라의 시점 차이와 초기 영상의 재현 여부를 기록합니다.

**이미지 경로를 바꾸는 것만으로는 reset이 실행되지 않습니다.** `Image(...)`는 이미 저장된 파일을 열기만 합니다. `run_lesson(...)`이 먼저 정상 종료되어 PNG가 생성되어야 합니다.

### 1. 비교 결과를 생성하고 표시하기

1. 위 설정 셀과 예제 셀을 순서대로 실행하여 `inspect-01`의 두 이미지를 확인합니다. 예제의 이름은 그대로 둡니다.
2. 아래 셀 전체를 실행합니다. `comparison_name` 하나를 실행과 표시 경로에 함께 사용하므로 경로를 따로 고치지 않습니다.
3. 실행이 끝나 두 이미지가 표시될 때까지 기다립니다. 오류가 나면 이미지 표시로 건너뛰지 말고 먼저 발생한 오류를 확인합니다.
4. 첫 비교에서는 `seed=7`과 `initial_state=0`을 모두 유지합니다. `SESSION`도 바꾸지 않습니다.

`inspect-02`는 **결과 폴더의 이름** 이며 seed나 초기 배치 번호가 아닙니다. 처음 쓰는 이름이면 실제 reset을 실행합니다. 같은 조건의 완료 결과가 이미 있으면 저장된 결과를 읽으므로 reset을 다시 실행하지 않습니다. 새 reset을 더 수행하려면 `inspect-02-repeat`처럼 사용하지 않은 이름으로 바꿉니다.


In [ ]:
# TODO 1: 아래 셀 전체를 실행하여 같은 초기조건의 비교 결과를 만드세요.
comparison_name = 'inspect-02'
second = run_lesson('inspect', comparison_name, seed=7, initial_state=0)

# 실행에 사용한 이름으로 표시 경로도 자동으로 맞춥니다.
from IPython.display import display, Image
comparison_dir = RUNS / comparison_name
image_paths = [comparison_dir / 'image.png', comparison_dir / 'image2.png']
missing = [str(path) for path in image_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        '비교 이미지가 없습니다: ' + ', '.join(missing)
        + '\n위 실행 로그를 확인하세요. 기존 완료 결과의 PNG가 누락되었다면 '
        'comparison_name을 사용하지 않은 이름으로 바꾸고 이 셀 전체를 다시 실행하세요.'
    )
for path in image_paths:
    print(path.name, '—', path)
    display(Image(filename=str(path)))
second


### 2. 관찰 답안 작성하기

**TODO 2:** 아래 `notes` 셀에 자신의 관찰 결과를 작성하고 노트북을 저장하세요. 새 정책 함수를 구현하는 과제가 아니라, 조건을 통제하고 결과를 비교하는 실습입니다.

| 비교 목적 | 비교할 영상 | 적을 내용 |
|---|---|---|
| 카메라 시점 차이 | `inspect-02/image.png` ↔ `inspect-02/image2.png` | 같은 물체가 어디에 보이는지, 크기·가림이 어떻게 다른지 |
| 같은 초기조건의 재현 여부 | `inspect-01/image.png` ↔ `inspect-02/image.png`, 이어 각 `image2.png` | 같은 카메라에서 물체·로봇의 초기 위치가 같은지, 다른 부분이 있는지 |

권장 기록 형식입니다. **관찰하지 않은 결과는 `미측정`으로 남깁니다.** 육안으로 같아 보인다는 판단과 픽셀 값이 완전히 같다는 검증은 구분하세요.

```python
notes = {
    'baseline_run': 'inspect-01',
    'comparison_run': comparison_name,
    'seed': 7,
    'initial_state': 0,
    'camera_difference': '미측정',
    'same_seed_comparison': '미측정',
    'evidence': '비교한 파일명과 실제 관찰한 물체 위치·가림을 작성',
}
```

### 3. 초기 배치만 바꾸어 비교하기

첫 비교와 답안 작성을 마친 뒤, 위 비교 실행 셀의 `comparison_name`을 `inspect-03`으로, `initial_state=0`을 `initial_state=1`로 바꾸어 **셀 전체를 다시 실행** 합니다. seed는 7로 유지합니다. 이번에는 초기 배치만 바꾼 비교입니다. 기준 영상과 물체 위치 및 결과 JSON의 task가 어떻게 달라지거나 유지되는지 확인하고 `notes`에 별도 항목으로 기록합니다. 첫 비교의 관찰 기록은 남겨 두세요.

### `FileNotFoundError`가 나면

- 표시 경로만 `inspect-02`로 바꿨다면, 위 **비교 결과 생성·표시 셀 전체** 를 실행하세요.
- reset 실행 중 오류가 났다면, 먼저 해당 실행의 오류를 해결해야 합니다. 빈 폴더를 만들거나 기존 PNG를 복사해 채우면 재현 비교가 되지 않습니다.
- `기존 completed 결과 읽기`가 출력되는데 PNG가 없다면, 결과 JSON만 남은 불완전한 자료일 수 있습니다. 기존 자료를 보존하고 새 실행 이름으로 다시 생성하세요.
- 커널을 재시작했거나 `SESSION`을 바꿨다면, 설정 셀부터 실행하고 기준 `inspect-01`도 현재 회차에 있는지 확인하세요.


In [ ]:
# 관찰 결과/실험 설정을 여기에 기록하세요.
notes = {}
notes

## 검증과 정답·복구

두 카메라는 외부 시점과 손목 시점을 제공합니다. result status=completed와 termination_reason=inspection_only를 확인합니다. success=false는 작업을 실행하지 않았다는 뜻입니다.

TODO 완료 예시(완료 결과가 있으면 같은 조건인지 확인하고 읽습니다):
```python
second = run_lesson('inspect', 'inspect-02')
```

파일·캐시·CUDA 오류는 [복구 안내](README.md#복구)와 이 노트북 시작 절에 있습니다. 필요한 복구는 전용 이미지·/opt/course 커널 확인, Data 마운트·배포 모델/캐시 존재 확인, 다른 GPU 작업 종료 후 한 프로세스 실행입니다. 누락 파일을 다운로드하거나 상태8을 잘라 우회하지 않습니다. CLI는 노트북과 같은 실험의 대체 실행입니다. 같은 조건의 완료 결과는 다시 읽습니다. 새 실험은 SESSION과 후속 노트북의 회차를 함께 바꾸세요. 실험 출력은 삭제하지 않습니다.

### 정답 해설 — 입력부터 결과까지

**reset 결과 읽기:** inspect-02라는 새 이름으로 같은 seed·initial_state를 요청하고 두 카메라를 각각 이전 실행의 같은 카메라와 비교합니다. 외부 시점과 손목 시점을 서로 같은 영상으로 기대하지 않습니다.

inspect는 작업을 수행하지 않으므로 status=completed·inspection_only와 success=false가 함께 나오는 것이 가능합니다. 단순한 성공/실패 문자열 대신 mode와 종료 이유를 읽습니다. 초기화는 잠시 걸릴 수 있으므로 로그와 출력 폴더를 확인합니다.

**확인 질문:** seed만 바꾸는 것과 initial_state를 바꾸는 것은 어떤 조건을 각각 바꾸나요?

**적용·재확인:** 관찰 답안은 자신의 실제 파일·조건으로 작성합니다. 완료된 동일 조건 결과는 다시 읽고, 새 실험은 새 실행 이름을 사용합니다. `SESSION`을 바꾸면 후속 노트북도 맞춥니다. 형상 검사·실행 완료·작업 성공을 따로 확인하며, 예시 수치를 자기 측정값으로 복사하지 않습니다.

<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 초기상태와 seed·두 카메라 관찰 결과 기록

[5일차 슬라이드 28쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s024) · [배포 PDF 28쪽](../슬라이드/5일차.pdf#page=28)로 돌아가 **28–31쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

